In [12]:
import csv
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

with open("wine_dataset.csv", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    features = [name for name in reader.fieldnames if name != "target"]

X = torch.tensor([[float(row[c]) for c in features] for row in rows], dtype=torch.float32)
y = torch.tensor([int(row["target"]) for row in rows], dtype=torch.long)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("classes:", torch.unique(y, return_counts=True))

X: torch.Size([178, 13]) torch.float32
y: torch.Size([178]) torch.int64
classes: (tensor([0, 1, 2]), tensor([59, 71, 48]))


In [13]:
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=y.numpy(), random_state=42
)
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("train/test:", len(train_idx), len(test_idx))

train/test: 142 36


In [14]:
def standardize(train_x, other_x):
    scaler = StandardScaler()
    xtr = scaler.fit_transform(train_x.numpy())
    xother = scaler.transform(other_x.numpy())
    return torch.tensor(xtr, dtype=torch.float32), torch.tensor(xother, dtype=torch.float32)

xtr, xte = standardize(X_train, X_test)
print("scaled:", xtr.shape, xte.shape)

scaled: torch.Size([142, 13]) torch.Size([36, 13])


In [15]:
def fit(train_x, train_y, epochs=250, seed=42):
    torch.manual_seed(seed)
    model = nn.Linear(train_x.shape[1], 3)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

    loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=16,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    return model

In [16]:
@torch.no_grad()
def accuracy(model, x, y_true):
    model.eval()
    predictions = model(x).argmax(dim=1)
    return accuracy_score(y_true.numpy(), predictions.numpy())

In [17]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
dummy_scores = []

for fold_number, (tr_idx, val_idx) in enumerate(
    skf.split(X_train.numpy(), y_train.numpy()), start=1
):
    xtr, xval = standardize(X_train[tr_idx], X_train[val_idx])

    model = fit(xtr, y_train[tr_idx])
    score = accuracy(model, xval, y_train[val_idx])
    cv_scores.append(score)

    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train[tr_idx].numpy(), y_train[tr_idx].numpy())
    baseline = dummy.score(X_train[val_idx].numpy(), y_train[val_idx].numpy())
    dummy_scores.append(baseline)

    print(f"Fold {fold_number}: PyTorch={score:.3f} | Dummy={baseline:.3f}")

print(f"CV accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"Dummy CV accuracy: {np.mean(dummy_scores):.4f}")

Fold 1: PyTorch=0.931 | Dummy=0.414
Fold 2: PyTorch=1.000 | Dummy=0.414
Fold 3: PyTorch=1.000 | Dummy=0.393
Fold 4: PyTorch=1.000 | Dummy=0.393
Fold 5: PyTorch=0.929 | Dummy=0.393
CV accuracy: 0.9719 ± 0.0344
Dummy CV accuracy: 0.4012


In [18]:
xtr, xte = standardize(X_train, X_test)
final_model = fit(xtr, y_train)
print(f"Final test accuracy: {accuracy(final_model, xte, y_test):.4f}")

Final test accuracy: 0.9722
